# ArXiv RAG System — Evaluation Pipeline

Measures retrieval quality, generation quality, latency distribution, and citation accuracy against the locked success metrics.

**Prerequisite:** `01_rag_pipeline_development.ipynb` must have been run first — ChromaDB index must be populated.

---

| Section | What is measured |
|---------|------------------|
| A | Retrieval — Precision@5, Recall@10, MRR, Hit Rate@5 |
| B | Generation — RAGAS, ROUGE-L, citation accuracy |
| C | Latency — P50, P90, P95, P99 end-to-end |
| D | Analysis — per-category breakdown, failure cases |
| E | Report — scorecard vs. targets, save JSON |


## 0. Setup

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

BASE = Path('/kaggle/working')
os.chdir(BASE)
sys.path.insert(0, str(BASE))

import yaml
import numpy as np
import pandas as pd

config = yaml.safe_load((BASE / 'config.yaml').read_text())
print('Config loaded ✅')
print(f"  vector store : {config['vector_store']['type']}")
print(f"  embedding    : {config['embedding']['model_name']}")
print(f"  llm mode     : {config['llm']['mode']}")

In [ ]:
# ── Load all pipeline components ──────────────────────────────────────
from src.embedding_pipeline import EmbeddingModel, SemanticChunker
from src.vector_store       import ChromaVectorStore
from src.retrieval          import DocumentRetriever
from src.generation         import LLMBackend, PromptBuilder, CitationFormatter, RAGPipeline
from src.evaluation         import (
    RetrievalEvaluator, LatencyEvaluator, CitationEvaluator,
    EvaluationReport, RetrievalMetrics, GenerationMetrics, LatencyMetrics,
)

# Embedding model
embedder = EmbeddingModel(
    model_name=config['embedding']['model_name'],
    device=config['embedding']['device'],
)
embedder.load()
print(f'Embedder : {config["embedding"]["model_name"]} | dim={embedder.dimensions}')

# Vector store
chroma_dir   = str(BASE / config['vector_store']['chroma_persist_dir'].lstrip('./'))
vector_store = ChromaVectorStore(
    persist_directory=chroma_dir,
    collection_name='arxiv_papers',
)
vector_store.connect()
index_size = vector_store.count()
print(f'Index    : {index_size:,} chunks')
assert index_size >= 10_000, f'Index too small: {index_size}'

# Retriever
retriever = DocumentRetriever(
    vector_store=vector_store,
    embedder=embedder,
    top_k=config['retrieval']['top_k'],
    score_threshold=config['retrieval']['score_threshold'],
    mmr_lambda=config['retrieval']['mmr_lambda'],
)

# LLM + RAG pipeline
llm = LLMBackend(config=config)
llm.load()
print(f'LLM      : {llm.backend_name}')

rag = RAGPipeline(
    retriever=retriever,
    llm=llm,
    prompt_builder=PromptBuilder(config=config),
    citation_formatter=CitationFormatter(),
)
print('All components loaded ✅')

## Test Set Construction

500 questions generated from 100 sampled papers (5 per paper). Each question is paired with its source `paper_id` as ground truth for retrieval evaluation.

In [ ]:
TEST_SET_PATH = BASE / 'data/processed/eval_test_set.json'

if TEST_SET_PATH.exists():
    test_set = json.loads(TEST_SET_PATH.read_text())
    print(f'Loaded existing test set: {len(test_set)} questions')
else:
    # ── Generate test questions from indexed papers ────────────────────
    df = pd.read_parquet(BASE / 'data/processed/arxiv_clean.parquet')
    sample_papers = df.sample(n=100, random_state=42)

    QUESTION_TEMPLATES = [
        'What is the main contribution of the paper about {topic}?',
        'How does {topic} work according to recent research?',
        'What methods are proposed for {topic}?',
        'What are the key findings related to {topic}?',
        'What datasets or benchmarks are used to evaluate {topic}?',
    ]

    test_set = []
    for _, row in sample_papers.iterrows():
        # Extract topic from title (first 4 words)
        topic = ' '.join(str(row.get('title', '')).split()[:4])
        for template in QUESTION_TEMPLATES:
            test_set.append({
                'query':               template.format(topic=topic),
                'relevant_paper_ids':  [str(row['paper_id'])],
                'category':            str(row.get('category', '')),
                'source_title':        str(row.get('title', '')),
            })

    TEST_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
    TEST_SET_PATH.write_text(json.dumps(test_set, indent=2))
    print(f'Generated and saved {len(test_set)} test questions')

print(f'\nTest set sample:')
for item in test_set[:3]:
    print(f"  Q: {item['query']}")
    print(f"  A: {item['relevant_paper_ids']}")
    print()

## Section A — Retrieval Evaluation

Measures whether the vector store retrieves the source paper for each generated question. Ground truth = `relevant_paper_ids` from test set.

In [ ]:
print('Running retrieval evaluation...')
print(f'Test queries: {len(test_set)}')
print(f'top_k       : {config["retrieval"]["top_k"]}')
print()

ret_evaluator  = RetrievalEvaluator()
ret_metrics    = ret_evaluator.evaluate(
    retriever=retriever,
    test_queries=test_set,
    top_k=config['retrieval']['top_k'],
)

TARGETS = {
    'Precision@5':  0.80,
    'Recall@10':    0.75,
    'MRR':          0.70,
    'Hit Rate@5':   0.90,
}

achieved = {
    'Precision@5': ret_metrics.precision_at_5,
    'Recall@10':   ret_metrics.recall_at_10,
    'MRR':         ret_metrics.mrr,
    'Hit Rate@5':  ret_metrics.hit_rate_at_5,
}

print('Retrieval results:')
print('─' * 55)
print(f'  {"Metric":<20} {"Target":>8} {"Achieved":>10} {"Status":>8}')
print('─' * 55)
for metric, target in TARGETS.items():
    val    = achieved[metric]
    status = '✅' if val >= target else '❌'
    print(f'  {metric:<20} {target:>8.2f} {val:>10.4f} {status:>8}')
print('─' * 55)
print(f'  Queries evaluated: {ret_metrics.num_queries}')

In [ ]:
# ── Per-category retrieval breakdown ──────────────────────────────────
categories = list(config['indexing']['categories'].keys())
print('Per-category Hit Rate@5:')
print('─' * 40)
for cat in categories:
    cat_queries = [q for q in test_set if q.get('category') == cat]
    if not cat_queries:
        continue
    cat_metrics = ret_evaluator.evaluate(
        retriever=retriever,
        test_queries=cat_queries,
        top_k=5,
    )
    bar = '█' * int(cat_metrics.hit_rate_at_5 * 20)
    print(f'  {cat:<10} {cat_metrics.hit_rate_at_5:.3f}  {bar}')

## Section B — Generation Evaluation

Evaluates citation accuracy across the full test set. RAGAS faithfulness and answer relevancy are computed on a 50-question subset (full 500 would require significant LLM compute).

In [ ]:
# ── Citation accuracy — full test set ─────────────────────────────────
print('Running citation evaluation on 100-question sample...')
EVAL_SAMPLE = test_set[:100]

responses = []
for i, item in enumerate(EVAL_SAMPLE):
    if i % 10 == 0:
        print(f'  Progress: {i}/{len(EVAL_SAMPLE)}')
    try:
        resp = rag.query(question=item['query'])
        responses.append({
            'answer':    resp.answer,
            'citations': resp.citations,
            'sources':   [
                {'citation_str': s.citation_str}
                for s in resp.sources
            ],
            'query':     item['query'],
            'latency_ms': resp.total_latency_ms,
        })
    except Exception as exc:
        print(f'  ⚠️  Query {i} failed: {exc}')

cit_evaluator = CitationEvaluator()
citation_acc  = cit_evaluator.evaluate(responses)
print(f'\nCitation accuracy : {citation_acc:.4f}')
status = '✅' if citation_acc >= 0.95 else '❌'
print(f'Target ≥ 0.95     : {status}')

In [ ]:
# ── ROUGE-L on sample answers ─────────────────────────────────────────
try:
    from rouge_score import rouge_scorer
    scorer    = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    # Use first response answer vs second response answer as proxy
    # (no human reference available — ROUGE-L between RAG answers)
    # Real evaluation would use human-written reference answers
    rouge_scores = []
    for i in range(min(50, len(responses) - 1)):
        score = scorer.score(
            responses[i]['answer'],
            responses[i+1]['answer'],
        )
        rouge_scores.append(score['rougeL'].fmeasure)
    avg_rouge = float(np.mean(rouge_scores))
    print(f'ROUGE-L (cross-answer, indicative): {avg_rouge:.4f}')
    print('Note: True ROUGE-L requires human reference answers.')
except Exception as exc:
    print(f'ROUGE-L skipped: {exc}')
    avg_rouge = 0.0

In [ ]:
# ── RAGAS evaluation on 50-question subset ────────────────────────────
ragas_faithfulness   = 0.0
ragas_relevancy      = 0.0

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    from datasets import Dataset

    RAGAS_SAMPLE = test_set[:50]
    ragas_data   = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

    print('Building RAGAS dataset (50 questions)...')
    for item in RAGAS_SAMPLE:
        try:
            resp = rag.query(question=item['query'])
            ragas_data['question'].append(item['query'])
            ragas_data['answer'].append(resp.answer)
            ragas_data['contexts'].append(
                [s.text for s in resp.sources]
            )
            ragas_data['ground_truth'].append(item['query'])  # proxy
        except Exception:
            continue

    if len(ragas_data['question']) > 0:
        dataset        = Dataset.from_dict(ragas_data)
        ragas_result   = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
        ragas_faithfulness = float(ragas_result['faithfulness'])
        ragas_relevancy    = float(ragas_result['answer_relevancy'])
        print(f'RAGAS Faithfulness   : {ragas_faithfulness:.4f}')
        print(f'RAGAS Ans Relevancy  : {ragas_relevancy:.4f}')
    else:
        print('RAGAS: no valid responses collected')

except Exception as exc:
    print(f'RAGAS skipped (library/API issue): {exc}')
    print('Setting placeholder values — re-run with valid RAGAS setup.')

## Section C — Latency Evaluation

Times 100 end-to-end queries (retrieval + generation) and computes the P50/P90/P95/P99 distribution. Target: P95 < 2,000ms.

In [ ]:
print('Running latency evaluation (100 queries)...')
lat_evaluator = LatencyEvaluator()
lat_queries   = [item['query'] for item in test_set[:100]]
lat_metrics   = lat_evaluator.evaluate(rag, lat_queries)

print(f'\nLatency distribution ({lat_metrics.num_queries} queries):')
print(f'  P50  : {lat_metrics.p50_ms:>8.1f} ms')
print(f'  P90  : {lat_metrics.p90_ms:>8.1f} ms')
print(f'  P95  : {lat_metrics.p95_ms:>8.1f} ms  {"✅" if lat_metrics.p95_ms < 2000 else "❌"} (target < 2000ms)')
print(f'  P99  : {lat_metrics.p99_ms:>8.1f} ms')
print(f'  Mean : {lat_metrics.mean_ms:>8.1f} ms')

In [ ]:
# ── Latency histogram ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

# Re-time to get raw values for plotting
latencies = []
for q in lat_queries:
    t0 = time.perf_counter()
    try:
        rag.query(question=q)
        latencies.append((time.perf_counter() - t0) * 1000)
    except Exception:
        pass

plots_dir = BASE / 'evaluation_plots'
plots_dir.mkdir(exist_ok=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(latencies, bins=20, color='steelblue', edgecolor='white')
ax1.axvline(np.percentile(latencies, 95), color='red',
            linestyle='--', label='P95')
ax1.axvline(2000, color='orange',
            linestyle='--', label='Target 2000ms')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Count')
ax1.set_title('End-to-End Latency Distribution')
ax1.legend()

sorted_lat = np.sort(latencies)
cdf        = np.arange(1, len(sorted_lat) + 1) / len(sorted_lat)
ax2.plot(sorted_lat, cdf, color='steelblue')
ax2.axvline(2000, color='orange', linestyle='--', label='2000ms target')
ax2.axhline(0.95,  color='red',    linestyle='--', label='P95')
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('CDF')
ax2.set_title('Latency CDF')
ax2.legend()

plt.tight_layout()
plt.savefig(plots_dir / 'latency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: evaluation_plots/latency_distribution.png')

## Section D — Failure Analysis

In [ ]:
# ── Low confidence responses ───────────────────────────────────────────
low_conf = [r for r in responses if '(low confidence)' in r.get('answer','').lower()
            or not r.get('citations')]
print(f'Responses with no citations / low confidence: {len(low_conf)}/{len(responses)}')

# ── Slowest queries ────────────────────────────────────────────────────
sorted_by_latency = sorted(responses, key=lambda r: r.get('latency_ms', 0), reverse=True)
print(f'\nTop 5 slowest queries:')
for r in sorted_by_latency[:5]:
    print(f"  {r.get('latency_ms',0):>8.1f}ms  {r['query'][:60]}")

# ── Hallucination check ────────────────────────────────────────────────
hallucinated = [
    r for r in responses
    if r.get('citations') and not all(
        any(c == s['citation_str'] for s in r.get('sources', []))
        for c in r['citations']
    )
]
print(f'\nHallucinated citations detected: {len(hallucinated)}/{len(responses)}')
for r in hallucinated[:3]:
    print(f"  Query: {r['query'][:60]}")
    print(f"  Citations: {r['citations']}")
    print(f"  Sources:   {[s['citation_str'] for s in r['sources']]}")

## Section E — Final Scorecard

In [ ]:
from datetime import datetime

report = EvaluationReport(
    retrieval=ret_metrics,
    generation=GenerationMetrics(
        faithfulness=ragas_faithfulness,
        answer_relevancy=ragas_relevancy,
        citation_accuracy=citation_acc,
        rouge_l=avg_rouge,
        num_evaluated=len(responses),
    ),
    latency=lat_metrics,
    model_used=llm.backend_name,
    vector_store=config['vector_store']['type'],
    index_size=index_size,
    timestamp=datetime.utcnow().isoformat(),
)

report.print_scorecard()

report_path = str(BASE / 'evaluation_report.json')
report.save(report_path)
print(f'\nReport saved to: {report_path}')

In [ ]:
# ── Confirm all target files exist ────────────────────────────────────
deliverables = [
    BASE / 'evaluation_report.json',
    BASE / 'evaluation_plots' / 'latency_distribution.png',
    BASE / 'data/processed/eval_test_set.json',
]
print('Evaluation deliverables:')
for p in deliverables:
    status = '✅' if p.exists() else '❌'
    size   = f'{p.stat().st_size:,}b' if p.exists() else 'missing'
    print(f'  {status}  {p.name:<40} {size}')